<a href="https://colab.research.google.com/github/aimldstejas/aibits-genai-notebooks/blob/main/course-1-deep-learning/lab-01-forward-pass-by-hand-and-by-code.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lab 1 (graded) — Forward pass, by hand and by code
**Course 1: Hands-On Deep Learning with Python — Chapter 1: A neuron, a layer, a network**

**Problem brief (Leo Farkas, Meridian Bank):** "Our gradient-boosted credit model is
accurate but we're told a neural network could pick up interactions we're missing. Prove
it's even worth trying."

**Dataset:** UCI Default of Credit Card Clients. **Offline fallback:** included below.

**What you'll submit:**
1. Working `Linear`, `ReLU`, `Sigmoid`, `BCELoss` classes and an `MLP` built from them.
2. A forward pass on real Meridian applicants, confirming the untrained loss is close to
   random (≈0.69, i.e. −ln 0.5).
3. A pencil-and-paper worked example for one neuron (markdown cell at the bottom).
4. Two short paragraphs: why the network can't do anything useful yet, and the scoping
   note Sam needs (is a neural net worth pursuing here, given the bank's existing GBM).

## 1. Load the data (with offline fallback)

In [ ]:
import io
import urllib.request
import numpy as np
import pandas as pd

np.random.seed(0)

def load_credit_data():
    try:
        # fetch with a bounded timeout first - pd.read_excel(url) has no timeout of its own
        # and can hang the whole cell indefinitely on a stalled connection
        req = urllib.request.Request(
            'https://archive.ics.uci.edu/ml/machine-learning-databases/00350/'
            'default%20of%20credit%20card%20clients.xls',
            headers={'User-Agent': 'aibits-course-lab/1.0'},
        )
        with urllib.request.urlopen(req, timeout=30) as resp:
            raw = resp.read()
        df = pd.read_excel(io.BytesIO(raw), header=1, index_col=0)
        print('Loaded the real UCI dataset:', df.shape)
        return df
    except Exception as e:
        print(f'Offline fallback engaged ({e}) — generating a small synthetic stand-in '
              'with the same columns and a similar signal, for pipeline development only.')
        n = 2000
        limit = np.random.lognormal(9.5, 0.6, n)
        age = np.random.randint(21, 70, n)
        bill = limit * np.random.uniform(0.1, 0.9, n)
        pay = bill * np.random.uniform(0.0, 1.0, n)
        risk = 1 / (1 + np.exp(-(-2 + 0.00002 * (bill - pay) - 0.00001 * limit)))
        default = (np.random.rand(n) < risk).astype(int)
        return pd.DataFrame({
            'LIMIT_BAL': limit, 'AGE': age, 'BILL_AMT1': bill, 'PAY_AMT1': pay,
            'default payment next month': default,
        })

df = load_credit_data()

feature_cols = [c for c in ['LIMIT_BAL', 'AGE', 'BILL_AMT1', 'PAY_AMT1'] if c in df.columns]
target_col = 'default payment next month'
X = df[feature_cols].to_numpy(dtype=np.float64)
y = df[target_col].to_numpy(dtype=np.float64).reshape(-1, 1)

# standardize features — an untrained-network sanity check should not be confounded by scale
X = (X - X.mean(axis=0)) / (X.std(axis=0) + 1e-8)
print('X:', X.shape, 'y:', y.shape, 'positive rate:', y.mean().round(3))

## 2. Build the layers from scratch
Fill in the `TODO`s. Each `forward` takes the layer's input and returns its output — no
`backward` yet, that's Chapter 2.

In [ ]:
class Linear:
    def __init__(self, n_in, n_out):
        self.W = np.random.randn(n_in, n_out) * 0.01
        self.b = np.zeros(n_out)

    def forward(self, x):
        # TODO: return x @ W + b
        raise NotImplementedError


class ReLU:
    def forward(self, x):
        # TODO: elementwise max(0, x)
        raise NotImplementedError


class Sigmoid:
    def forward(self, x):
        # TODO: 1 / (1 + exp(-x)) — clip x to avoid overflow, e.g. np.clip(x, -500, 500)
        raise NotImplementedError


class BCELoss:
    def forward(self, y_hat, y_true, eps=1e-8):
        # TODO: mean of -[y*log(y_hat) + (1-y)*log(1-y_hat)], clipping y_hat to [eps, 1-eps]
        raise NotImplementedError

## 3. Assemble the MLP

In [ ]:
class MLP:
    """sizes=[n_in, hidden1, hidden2, ..., n_out]; ReLU on hidden layers, Sigmoid on output."""
    def __init__(self, sizes):
        self.layers = []
        for n_in, n_out in zip(sizes[:-1], sizes[1:]):
            self.layers.append(Linear(n_in, n_out))
            self.layers.append(ReLU())
        self.layers[-1] = Sigmoid()  # replace the final ReLU with a Sigmoid output

    def forward(self, x):
        for layer in self.layers:
            x = layer.forward(x)
        return x


model = MLP([X.shape[1], 16, 16, 1])
loss_fn = BCELoss()

y_hat = model.forward(X)
loss = loss_fn.forward(y_hat, y)
print('Untrained forward-pass loss:', round(float(loss), 4), '(expect close to 0.6931 = -ln 0.5)')
assert 0.5 < loss < 0.9, 'Loss is far from the random baseline — check your Sigmoid/BCELoss.'
print('Sanity check passed: the untrained network is behaving like a coin flip, as expected.')

## 4. Pencil-and-paper worked example (fill in)
Pick one real applicant row (`X[0]`) and one neuron in the first hidden layer (`model.layers[0].W[:, 0]`,
`model.layers[0].b[0]`). By hand, compute `z = w·x + b` and `a = ReLU(z)`. Show your work here,
then verify it matches `model.layers[0].forward(X[:1])[0, 0]` (after ReLU) in the cell below.

In [ ]:
# Verify your by-hand computation against the code
z0 = model.layers[0].forward(X[:1])
a0 = model.layers[1].forward(z0)
print('z (first neuron, first applicant):', z0[0, 0])
print('a = ReLU(z):', a0[0, 0])

## 5. Write-up (fill in, ~2 paragraphs)
**(a) Why can't this network do anything useful yet?** _Your answer here._

**(b) Scoping note for Sam — is a neural net worth pursuing for Meridian, given the bank's
existing GBM?** _Your answer here — an honest "not yet, here's what would change my mind" is a
legitimate answer at this stage of the course._

---
*Beacon AI · AIBits Academy — Chapter 1: A neuron, a layer, a network*